# Day 6 · Lab 5: Adaptive RAG & Self-Correcting Recovery Loop
### Advanced RAG Curriculum · Enterprise AI Engineering · Udaan Learning



In [ ]:
from typing import TypedDict, List, Dict, Any, Optional
from langgraph.graph import StateGraph, START, END
from langchain_core.documents import Document

class AdaptiveState(TypedDict):
    question: str
    active_strategy: str
    documents: List[Document]
    evidence_found: bool
    retry_count: int
    max_retries: int
    answer: Optional[str]

def node_retrieve(state: AdaptiveState) -> Dict[str, Any]:
    if state["active_strategy"] == "sql":
        docs = [Document(page_content="EHR Record: Rahul Sharma prescribed Metformin 500mg BID")]
    else:
        docs = [Document(page_content="General hospital policy on diabetes care")]
    return {"documents": docs}

def node_evaluate(state: AdaptiveState) -> Dict[str, Any]:
    docs_text = " ".join([d.page_content for d in state["documents"]])
    sufficient = "500mg" in docs_text
    return {"evidence_found": sufficient}

def node_recovery(state: AdaptiveState) -> Dict[str, Any]:
    return {"active_strategy": "sql", "retry_count": state["retry_count"] + 1}

def node_generate(state: AdaptiveState) -> Dict[str, Any]:
    return {"answer": f"Verified Answer: {state['documents'][0].page_content}"}

def node_abstain(state: AdaptiveState) -> Dict[str, Any]:
    return {"answer": "Abstain: Record not found."}

workflow = StateGraph(AdaptiveState)
workflow.add_node("retrieve", node_retrieve)
workflow.add_node("evaluate", node_evaluate)
workflow.add_node("recovery", node_recovery)
workflow.add_node("generate", node_generate)
workflow.add_node("abstain", node_abstain)

workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "evaluate")
workflow.add_edge("recovery", "retrieve")
workflow.add_edge("generate", END)
workflow.add_edge("abstain", END)

def check_branch(state: AdaptiveState) -> str:
    if state["evidence_found"]:
        return "generate"
    elif state["retry_count"] < state["max_retries"]:
        return "retry"
    else:
        return "abstain"

workflow.add_conditional_edges("evaluate", check_branch, {
    "generate": "generate",
    "retry": "recovery",
    "abstain": "abstain"
})

app = workflow.compile()

# Test recovery from sub-optimal starting route
res = app.invoke({
    "question": "Rahul prescription",
    "active_strategy": "vector",
    "documents": [],
    "evidence_found": False,
    "retry_count": 0,
    "max_retries": 2,
    "answer": None
})

print("Recovery loop success! Final answer:", res["answer"])
